# Session comparison at 0.01°/s

Compare participant gains at 0.5 Hz across two Session 1 trials and one Session 2 trial. Use the same preprocessing as `2a_SmallRampAnalysis.ipynb` and t-based confidence intervals as `2b_GainAndAmplitudeAnalysis.ipynb`. The comparison is descriptive.


In [1]:
import sys
from pathlib import Path

# Work whether Jupyter starts in the project root or in notebooks/.
search_roots = [Path.cwd(), *Path.cwd().parents]
project_root = next(
    (root for root in search_roots if (root / 'src' / 'config.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        'Could not find the project root containing src/config.py. '
        'Open this notebook from inside the SmallRamp_Analysis project.'
    )
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import balancepy.anaropia as bpa
import balancepy.anaropia_project as bpap
from scipy.stats import t
from IPython.display import display

from src.config import paths

# Analysis settings

Conditions 2 and 3 are Session 1a and Session 1b; condition 14 is Session 2. All three use a stimulus velocity of 0.01°/s.


In [2]:
CONFIDENCE = 0.95
MAIN_FREQUENCY_HZ = 0.5
SESSION_CONDITIONS = {
    'Session 1a': [2],
    'Session 1b': [3],
    'Session 2': [14],
}
EXPECTED_TRIALS = {
    'Session 1a': 1,
    'Session 1b': 1,
    'Session 2': 1,
}

preproc_config = bpa.AnaropiaPreprocessingConfig()
preproc_config.end_time_seconds = 210
preproc_config.cut_to_cycles = True
preproc_config.cycle_length_samples = 180
preproc_config.cycle_start_samples = 878

sr_config = bpa.AnaropiaSRDataConfig(
    stimulus_name='stim_pitch',
    response_name=bpa.COM_LEGACY_AP,
    frequency_selection='all',
)

print(preproc_config)
print(sr_config)

AnaropiaPreprocessingConfig(
  samplingrate_Hz: 90
  resample: True
  end_time_seconds: 210
  filter_type: None
  filter_order: 2
  filter_cutoff_Hz: 8
  cut_to_cycles: True
  cycle_start_samples: 878
  cycle_length_samples: 180
)
AnaropiaSRDataConfig(stimulus_name='stim_pitch', response_name=COMConfig(shoulder_pos_column='shld_zpos', hip_pos_column='hip_zpos', shoulder_height_column='shld_ypos', hip_height_column='hip_ypos', rotation=True), frequency_selection='all', name=None, column_scales=None)


# Load and preprocess data

Keep participants with recordings in all three conditions and check that each has one recording per condition.


In [3]:
records = []

for session, conditions in SESSION_CONDITIONS.items():
    print(f'Loading {session} ...')
    for subject_id, condition, filepath, body_height_m in bpap.batch_iterator(
        paths=paths,
        conditions=conditions,
        output_columns=['body_height_m'],
    ):
        sr = bpa.build_sr_data(
            filename=filepath,
            body_height_m=body_height_m,
            sr_config=sr_config,
            preproc_config=preproc_config,
            name=f'{subject_id}_{condition}',
        )
        records.append({
            'subject_id': subject_id,
            'session': session,
            'condition': condition,
            'filepath': filepath,
            'sr': sr,
        })

record_index = pd.DataFrame([
    {
        'participant': record['subject_id'],
        'session': record['session'],
        'condition': record['condition'],
        'file': Path(record['filepath']).name,
    }
    for record in records
])
trial_counts = (
    record_index.groupby(['participant', 'session'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=SESSION_CONDITIONS, fill_value=0)
)

condition_participant_ids = [
    set(record_index.loc[record_index['session'].eq(session), 'participant'])
    for session in SESSION_CONDITIONS
]
paired_ids = sorted(set.intersection(*condition_participant_ids))
all_ids = set.union(*condition_participant_ids)
excluded_ids = sorted(all_ids - set(paired_ids))

print(f'Complete participant pairs: {len(paired_ids)}')
print(f'Excluded because one session is unavailable: {excluded_ids or "none"}')

paired_trial_counts = trial_counts.loc[paired_ids]
for session, expected in EXPECTED_TRIALS.items():
    observed = paired_trial_counts[session]
    if not observed.eq(expected).all():
        raise ValueError(
            f'Expected {expected} recording(s) per paired participant in {session}; '
            f'found:\n{observed[~observed.eq(expected)]}'
        )

display(trial_counts)


Loading Session 1a ...
Loading Session 1b ...
Loading Session 2 ...
Complete participant pairs: 12
Excluded because one session is unavailable: ['AL02ES09']


session,Session 1a,Session 1b,Session 2
participant,,,
AB12OC03,1,1,1
AL02ES09,1,1,0
AL28IE13,1,1,1
AU21AL26,1,1,1
BP13OG09,1,1,1
DO29DA06,1,1,1
KA07RI18,1,1,1
MA05IE07,1,1,1
NI26RE08,1,1,1


# Participant gains at 0.5 Hz

Calculate one gain per participant and trial by pooling the cross-spectrum and stimulus power across cleaned cycles.


In [4]:
def pooled_participant_gain(session_records, target_frequency_hz=0.5):
    """Calculate the main-analysis pooled FRF gain for one participant/session."""
    numerator = 0j
    denominator = 0.0
    actual_frequencies = []

    for record in session_records:
        sr = record['sr']
        frequency_index = int(np.argmin(np.abs(sr.freq - target_frequency_hz)))
        actual_frequency = float(sr.freq[frequency_index])
        if not np.isclose(actual_frequency, target_frequency_hz):
            raise ValueError(
                f'Expected {target_frequency_hz} Hz but nearest bin is '
                f'{actual_frequency} Hz for {Path(record["filepath"]).name}.'
            )
        actual_frequencies.append(actual_frequency)

        stimulus = np.atleast_2d(sr.stimulus_spectrum)
        response = np.atleast_2d(sr.response_spectrum)
        if stimulus.shape[0] == 1 and sr.stimulus_spectrum.ndim == 1:
            stimulus = stimulus.T
            response = response.T

        yi = stimulus[frequency_index]
        yo = response[frequency_index]
        numerator += np.sum(yo * np.conjugate(yi))
        denominator += np.sum(np.abs(yi) ** 2)

    if denominator <= 0:
        raise ValueError('The pooled stimulus-spectrum denominator is not positive.')

    return float(np.abs(numerator / denominator)), actual_frequencies


participant_rows = []
for subject_id in paired_ids:
    row = {'participant': subject_id}
    for session in SESSION_CONDITIONS:
        selected_records = [
            record for record in records
            if record['subject_id'] == subject_id and record['session'] == session
        ]
        gain, actual_frequencies = pooled_participant_gain(
            selected_records,
            target_frequency_hz=MAIN_FREQUENCY_HZ,
        )
        row[session] = gain
        row[f'{session} trials'] = len(selected_records)
        row[f'{session} frequency (Hz)'] = actual_frequencies[0]
    participant_rows.append(row)

participant_gain = pd.DataFrame(participant_rows)

display(participant_gain)


,participant,Session 1a,Session 1a trials,Session 1a frequency (Hz),Session 1b,Session 1b trials,Session 1b frequency (Hz),Session 2,Session 2 trials,Session 2 frequency (Hz)
0,AB12OC03,1.648247,1,0.5,7.699058,1,0.5,5.594742,1,0.5
1,AL28IE13,1.628013,1,0.5,2.096332,1,0.5,0.892100,1,0.5
2,AU21AL26,0.131794,1,0.5,2.106865,1,0.5,6.462755,1,0.5
3,BP13OG09,2.247241,1,0.5,0.682982,1,0.5,0.501151,1,0.5
4,DO29DA06,0.753957,1,0.5,1.124805,1,0.5,1.664252,1,0.5
5,KA07RI18,1.822153,1,0.5,1.066391,1,0.5,2.779352,1,0.5
6,MA05IE07,1.261064,1,0.5,0.091278,1,0.5,3.915256,1,0.5
7,NI26RE08,0.975584,1,0.5,1.716689,1,0.5,0.639965,1,0.5
8,SE10FY02,1.467966,1,0.5,2.283370,1,0.5,1.410954,1,0.5
9,SI28IT27,1.409072,1,0.5,0.372813,1,0.5,1.288762,1,0.5


# Mean gain with 95% confidence intervals

Calculate the mean participant gain and t-based interval for each trial: $\mathrm{mean} \pm t\,s/\sqrt{n}$. Use the same participants in all three summaries.


In [5]:
def mean_t_ci(values, confidence=0.95):
    """Arithmetic mean and two-sided t-based confidence interval."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    mean = values.mean() if n else np.nan
    if n > 1:
        standard_deviation = values.std(ddof=1)
        standard_error = standard_deviation / np.sqrt(n)
        margin = t.ppf(1 - (1 - confidence) / 2, df=n - 1) * standard_error
    else:
        standard_deviation = standard_error = margin = np.nan
    return {
        'n': n,
        'mean': mean,
        'standard_deviation': standard_deviation,
        'standard_error': standard_error,
        'ci_lower': mean - margin,
        'ci_upper': mean + margin,
    }


session_summary = pd.DataFrame([
    {'session': session, **mean_t_ci(participant_gain[session], CONFIDENCE)}
    for session in SESSION_CONDITIONS
])

display(session_summary)


,session,n,mean,standard_deviation,standard_error,ci_lower,ci_upper
0,Session 1a,12,1.492668,0.726978,0.209860,1.030769,1.954568
1,Session 1b,12,1.956595,2.023482,0.584129,0.670935,3.242254
2,Session 2,12,2.526054,2.013474,0.581240,1.246754,3.805354


# Plot session comparison

Grey lines connect each participant’s three trial gains. Colored markers and error bars show the trial means and 95% t-based confidence intervals.


In [6]:
fig = go.Figure()
session_x = ['Session 1a', 'Session 1b', 'Session 2']

for _, participant in participant_gain.iterrows():
    fig.add_trace(go.Scatter(
        x=session_x,
        y=[participant[session] for session in session_x],
        mode='lines+markers',
        line=dict(color='rgba(128, 128, 128, 0.42)', width=1.2),
        marker=dict(
            color='rgba(128, 128, 128, 0.62)',
            size=6,
            symbol='circle',
        ),
        name=participant['participant'],
        legendgroup='Participants',
        showlegend=False,
        hovertemplate=(
            f'Participant={participant["participant"]}<br>'
            'Session=%{x}<br>'
            'Gain=%{y:.4g}<extra></extra>'
        ),
    ))

session_colors = {
    'Session 1a': 'darkorange',
    'Session 1b': 'darkorange',
    'Session 2': 'darkgreen',
}
for _, summary in session_summary.iterrows():
    session = summary['session']
    color = session_colors[session]
    fig.add_trace(go.Scatter(
        x=[session],
        y=[summary['mean']],
        mode='markers',
        marker=dict(
            color=color,
            size=10,
            symbol='circle',
            line=dict(color='white', width=1),
        ),
        name=f'{session} mean',
        customdata=[[summary['ci_lower'], summary['ci_upper'], summary['n']]],
        error_y=dict(
            type='data',
            symmetric=False,
            array=[summary['ci_upper'] - summary['mean']],
            arrayminus=[summary['mean'] - summary['ci_lower']],
            thickness=1.5,
            width=5,
            color=color,
        ),
        hovertemplate=(
            'Session=%{x}<br>'
            'Mean gain=%{y:.4g}<br>'
            '95% t-CI=[%{customdata[0]:.4g}, %{customdata[1]:.4g}]<br>'
            'Participants=%{customdata[2]:.0f}<extra></extra>'
        ),
    ))

fig.update_xaxes(
    title_text='Session',
    categoryorder='array',
    categoryarray=session_x,
    showline=True,
    linecolor='black',
    linewidth=1,
    ticks='outside',
    tickfont=dict(size=15),
    showgrid=False,
    zeroline=False,
)
fig.update_yaxes(
    type='log',
    title_text='gain',
    title_standoff=12,
    showline=True,
    linecolor='black',
    linewidth=1,
    ticks='outside',
    tickfont=dict(size=15),
    showgrid=True,
    gridcolor='rgba(0, 0, 0, 0.10)',
    zeroline=False,
)
fig.update_layout(
    template='plotly_white',
    width=720,
    height=520,
    font=dict(family='Times New Roman', size=16, color='black'),
    margin=dict(l=100, r=160, t=45, b=90),
    legend=dict(
        x=1.02,
        y=1.0,
        xanchor='left',
        yanchor='top',
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='lightgrey',
        borderwidth=1,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
# Emit the figure through IPython's native display protocol. This avoids
# browser servers, external JavaScript downloads, and renderer detection.
display(fig)

# Save figure and tables

Save the figure as HTML and, when static export is available, PNG and SVG in `results/paper_figures`. Save participant gains, session summaries, and trial counts in `results/tables`.


In [7]:
EXPORT_STATIC_IMAGES = True

paper_figure_dir = paths.project_root / 'results' / 'paper_figures'
table_dir = paths.project_root / 'results' / 'tables'
paper_figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

html_path = paper_figure_dir / 'figure_session_effect_gain_0p01.html'
png_path = paper_figure_dir / 'figure_session_effect_gain_0p01.png'
svg_path = paper_figure_dir / 'figure_session_effect_gain_0p01.svg'
participant_path = table_dir / 'session_effect_gain_0p01_participants.csv'
summary_path = table_dir / 'session_effect_gain_0p01_summary.xlsx'

fig.write_html(html_path, include_plotlyjs=True, full_html=True)
participant_gain.to_csv(participant_path, index=False)
with pd.ExcelWriter(summary_path) as writer:
    session_summary.to_excel(writer, sheet_name='Session summaries', index=False)
    participant_gain.to_excel(writer, sheet_name='Participant values', index=False)
    trial_counts.reset_index().to_excel(writer, sheet_name='Trial counts', index=False)

if EXPORT_STATIC_IMAGES:
    try:
        fig.write_image(png_path, width=1440, height=1040, scale=2)
        fig.write_image(svg_path, format='svg')
        print(f'Saved: {png_path}')
        print(f'Saved: {svg_path}')
    except Exception as error:
        print(f'Static image export skipped: {error}')
else:
    print('Static PNG/SVG export disabled (set EXPORT_STATIC_IMAGES = True to enable).')

print(f'Saved: {html_path}')
print(f'Saved: {participant_path}')
print(f'Saved: {summary_path}')